# Анимация детерминированной динамики SIR

**Авторы:** А. В. Королькова, PhD, Кулябов Д. С., DSc

**Принадлежность:** Российский университет дружбы народов

## Назначение скрипта

Данный скрипт создаёт GIF-анимацию, показывающую, как со временем меняется
количество людей в каждой из трёх групп (S, I, R) в модели SIR.

### Что делает скрипт

1. Выполняет детерминированную симуляцию модели SIR с фиксированными параметрами

2. Создаёт серию кадров (столбчатых диаграмм) для каждого момента времени

3. Объединяет кадры в GIF-анимацию

4. Сохраняет анимацию в файл

### Цель анимации

Анимация позволяет наглядно увидеть:
- распространение эпидемии во времени
- момент пика заболеваемости
- спад эпидемии
- переход популяции из состояния S в I, затем в R

## Параметры модели

| Параметр | Значение | Описание |
|----------|----------|----------|
| β | 0.3 | Коэффициент заражения |
| γ | 0.1 | Коэффициент выздоровления |
| tmax | 100.0 | Время симуляции |
| saveat | 0.2 | Шаг сохранения для плавности анимации |
| S₀ | 990 | Начальное число восприимчивых |
| I₀ | 10 | Начальное число инфицированных |
| R₀ | 0 | Начальное число выздоровевших |

## Выходные данные

| Файл | Описание |
|------|----------|
| `plots/sir_animation.gif` | GIF-анимация динамики S, I, R — рис. 6.4 |

## Интерпретация результатов

- **На первых кадрах:** I растёт, S падает

- **В момент пика:** I достигает максимума

- **Затем:** I снижается, R растёт

- **Финал:** I → 0, R → плато, S → остаточная восприимчивая популяция

Анимация даёт динамическое представление о том, как волна инфекции
проходит через популяцию.

## Инициализация проекта DrWatson

In [ ]:
using DrWatson
@quickactivate "project"

## Загрузка модуля SIRPetri и утилит

In [ ]:
include(srcdir("SIRPetri.jl"))
using .SIRPetri
using DataFrames, CSV, Plots

## Подключение функциональности для создания анимации

In [ ]:
using Plots

## Задание параметров симуляции

Для анимации используется шаг сохранения 0.2, чтобы обеспечить плавность
при воспроизведении (всего 500 кадров за 100 единиц времени).

In [ ]:
β = 0.3
γ = 0.1
tmax = 100.0

## Детерминированная симуляция

Создание сети Петри и выполнение ODE-симуляции.
Метод решения: Tsit5 (Рунге-Кутта 5-го порядка).

In [ ]:
net, u0, states = build_sir_network(β, γ)

df = simulate_deterministic(net, u0, (0.0, tmax), saveat = 0.2, rates = [β, γ])

## Создание анимации

### Принцип построения анимации

1. Для каждого момента времени из `df.time` создаётся отдельный кадр

2. Каждый кадр — столбчатая диаграмма (bar plot) значений S, I, R

3. Заголовок кадра отображает текущее время

4. Все кадры объединяются в GIF с помощью `animate`

In [ ]:
anim = @animate for i in 1:size(df, 1)
    title_time = round(df.time[i], digits = 1)

    bar(
        ["S", "I", "R"],
        [df.S[i], df.I[i], df.R[i]],
        title = "SIR Model at t = $title_time",
        ylabel = "Population",
        ylims = (0, 1000),
        color = [:blue :red :green],
        legend = false,
        bar_width = 0.6
    )
end

## Сохранение анимации

Параметры GIF:
- Частота кадров: 10 fps
- Общая длительность: 50 секунд (500 кадров)

**Рисунок 6.4:** Динамика развития эпидемии

In [ ]:
gif(anim, plotsdir("sir_animation.gif"), fps = 10)

## Завершение работы

Скрипт успешно выполнил:
- Детерминированную симуляцию (β=0.3, γ=0.1, tmax=100.0)
- Создание анимации (500 кадров)
- Сохранение GIF-файла → `plots/sir_animation.gif`

In [ ]:
println("Анимация сохранена в plots/sir_animation.gif")